In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "sumy", "bert-score", "sacrebleu"], check=True)

import os
import math
import numpy as np
import pandas as pd
import torch
import chardet
import evaluate
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")  # для Colab без GUI

# sumy — для экстрактивных методов
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer as SumyTokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
from sumy.summarizers.lex_rank import LexRankSummarizer   # PageRank на графе предложений
from sumy.summarizers.lsa import LsaSummarizer
from sumy.nlp.stemmers import Stemmer as SumyStemmer
from sumy.utils import get_stop_words

from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/ruT5_science_sum_gazeta"
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "overfit_checkpoints")
FINAL_MODEL_DIR = os.path.join(PROJECT_DIR, "overfit_final_model")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

model_checkpoint = "IlyaGusev/rut5_base_sum_gazeta"

max_input_length = 600
max_target_length = 200

device = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))